# Lecture 9 · Notebook 1 — Evaluating a small model on medical exam questions
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture09/lecture09_1_medical_qa_eval.ipynb)

**Companion to [Chapter 3 of the video lecture](https://nlp-course-cuhksz.github.io/lecture09.html#p3-s36)** (how to evaluate a medical LLM) and
**[Chapter 6](https://nlp-course-cuhksz.github.io/lecture09.html#p6-s81)** (verifiable medical problems for HuatuoGPT-o1). Exam benchmarks such as CMB and MedQA are the cheapest,
most reproducible way to test medical knowledge, but the lecture warns that a single accuracy number hides a lot. In this notebook you
will evaluate a 0.5B-parameter open model on real USMLE-style questions from **MedQA** and see:

1. how to score multiple-choice questions from the model's **letter probabilities** (no answer parsing needed),
2. how much the score moves when you merely **shuffle the order of the options** (position bias),
3. whether the model's **confidence** tells you when it is right (calibration), and
4. what happens when you **remove the options** and ask for an open answer, as HuatuoGPT-o1 does, and why you then need a **verifier**.

Runs on a free CPU runtime in about 10 minutes (a T4 GPU takes 2–3 minutes).

> ⚠️ **Not medical advice.** This notebook studies how language models *behave* on medical test questions. The model is tiny and often
> wrong; nothing it writes here is a diagnosis or a treatment recommendation. For health questions, ask a qualified clinician.

In [ ]:
!pip -q install -U transformers

In [ ]:
import torch, json, random, re, time, collections
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # ~1 GB; try "Qwen/Qwen2.5-1.5B-Instruct" on a T4 GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()
print("device:", device, "| parameters:", f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M")

## 0 · The data: MedQA (USMLE, 4 options)
MedQA collects multiple-choice questions in the style of the US Medical Licensing Examination (Jin et al., 2020). We use the 4-option
version of the official **test split** (1,273 questions) and draw a fixed random sample so the notebook stays fast.

In [ ]:
from huggingface_hub import hf_hub_download
path = hf_hub_download("GBaker/MedQA-USMLE-4-options", "phrases_no_exclude_test.jsonl", repo_type="dataset")
rows = [json.loads(l) for l in open(path, encoding="utf-8")]
random.seed(0)
N = 80                                   # raise to 300+ on a GPU for tighter error bars
items = random.sample(rows, N)
L = ["A", "B", "C", "D"]
print(len(rows), "test questions; using", N)
ex = items[0]
print("\n" + ex["question"]); [print(f"  {k}. {v}") for k, v in ex["options"].items()]; print("gold:", ex["answer_idx"])

## 1 · Multiple choice: read the answer off the letter probabilities
We show the question and options, ask for the letter, and look at the model's next-token distribution restricted to the four letters
`A B C D`. The argmax is the prediction, and the probability is the model's confidence. This is how most exam benchmarks are scored:
it avoids parsing free text, and it is deterministic.

In [ ]:
LETTER_IDS = [tok.encode(x, add_special_tokens=False)[0] for x in L]
SYSTEM = "You are taking a medical licensing exam."

def mc_prompt(question, options):
    body = question + "\n\n" + "\n".join(f"{k}. {v}" for k, v in zip(L, options)) + "\n\nAnswer with the letter of the correct option only."
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": body}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def letter_probs(question, options):
    x = tok(mc_prompt(question, options), return_tensors="pt").to(device)
    logits = model(**x).logits[0, -1, LETTER_IDS].float()
    return logits.softmax(-1).cpu().numpy()

t0 = time.time()
probs = np.array([letter_probs(it["question"], [it["options"][k] for k in L]) for it in items])
gold = np.array([L.index(it["answer_idx"]) for it in items])
pred = probs.argmax(1)
acc = (pred == gold).mean()
se = np.sqrt(acc * (1 - acc) / N)
print(f"accuracy {acc:.1%}  (95% CI ±{1.96*se:.1%}, chance 25%)   [{time.time()-t0:.0f}s]")

With 80 questions the 95% confidence interval is about ±10 points, so two models that differ by 5 points on a sample like this are
**not** distinguishable. Small benchmarks need error bars, a point the lecture makes about the 100-question pairwise evaluation (slide 28).

## 2 · Position bias: does the order of the options matter?
The correct answers in MedQA are spread roughly evenly over A–D. If the model's predictions are not, it prefers some *positions*
regardless of content. A direct test: rotate the options (A→B→C→D→A) four times and ask the same question again. A model that
"knows" the answer should pick the same option *text* every time.

In [ ]:
print("gold letters     :", dict(collections.Counter(L[g] for g in gold)))
print("predicted letters:", dict(collections.Counter(L[p] for p in pred)))

def rotate(options, s):
    return options[s:] + options[:s]

picks = []       # picks[i][s] = option text chosen under rotation s
t0 = time.time()
for it in items:
    opts = [it["options"][k] for k in L]
    row = []
    for s in range(4):
        o = rotate(opts, s)
        row.append(o[int(letter_probs(it["question"], o).argmax())])
    picks.append(row)
acc_by_rot = [np.mean([row[s] == it["answer"] for row, it in zip(picks, items)]) for s in range(4)]
consistent = np.mean([len(set(row)) == 1 for row in picks])
always = np.mean([all(p == it["answer"] for p in row) for row, it in zip(picks, items)])
print(f"accuracy under the 4 rotations: {[f'{a:.0%}' for a in acc_by_rot]}   [{time.time()-t0:.0f}s]")
print(f"same answer under all 4 orders: {consistent:.0%}")
print(f"correct under all 4 orders    : {always:.0%}")
plt.figure(figsize=(6, 3))
plt.bar(["rot 0", "rot 1", "rot 2", "rot 3", "consistent\n& correct"], acc_by_rot + [always], color=["#4c72b0"] * 4 + ["#c44e52"])
plt.axhline(0.25, ls="--", c="gray"); plt.text(4.4, 0.26, "chance", ha="right", color="gray")
plt.ylabel("fraction of questions"); plt.ylim(0, 1); plt.title("Same questions, options in a different order"); plt.show()

Two lessons. First, the accuracy itself moves with nothing but the order of the options, so a leaderboard that uses one fixed order
partly measures luck. Second, the share of questions the model gets right **under every order** is much lower than its accuracy: that is
a stricter, more honest number. It is the same idea as the order-swapping trick in HuatuoGPT's reward model (slide 23): only trust a
judgment that survives swapping.

## 3 · Does the model know when it does not know?
A trustworthy medical AI must be *calibrated* (slide 108): its confidence should match how often it is right. Split the questions by the
model's confidence and compare accuracy.

In [ ]:
conf = probs.max(1)
order = np.argsort(conf)
bins = np.array_split(order, 4)
for b, name in zip(bins, ["lowest 25%", "2nd quarter", "3rd quarter", "highest 25%"]):
    print(f"confidence {name:12s} mean conf {conf[b].mean():.2f}   accuracy {(pred[b] == gold[b]).mean():.0%}")
print(f"\noverall: mean confidence {conf.mean():.2f} vs accuracy {acc:.2f}")

If accuracy rises with confidence, the confidence carries *some* signal (you could refer the least confident cases to a human). If the
mean confidence is far from the accuracy, the model is mis-calibrated. Chain of Diagnosis (slide 103) uses exactly such a confidence to
decide whether to answer or to ask the patient another question.

## 4 · Remove the options: open-ended answers need a verifier
HuatuoGPT-o1 (slides 81–82) turns exam questions into **open-ended** ones: without the options, a model cannot eliminate its way to the
answer or guess with 25% luck. Let's ask our model for a short free-text answer and grade it.

In [ ]:
@torch.no_grad()
def open_answer(question, max_new_tokens=24):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": question + "\n\nGive only the final answer, in a few words."}]
    x = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors="pt").to(device)
    out = model.generate(**x, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def norm(s):
    return re.sub(r"[^a-z0-9 ]", " ", s.lower()).split()

def token_f1(a, b):
    a, b = norm(a), norm(b)
    common = sum((collections.Counter(a) & collections.Counter(b)).values())
    if common == 0: return 0.0
    p, r = common / len(a), common / len(b)
    return 2 * p * r / (p + r)

M = 30
t0 = time.time()
open_res = []
for it in items[:M]:
    ans = open_answer(it["question"])
    open_res.append((it["answer"], ans, token_f1(ans, it["answer"])))
print(f"[{time.time()-t0:.0f}s]")
for g, a, f in open_res[:10]:
    print(f"F1 {f:.2f} | gold: {g[:45]:45s} | model: {a[:60]}")
exact = np.mean([norm(g) == norm(a) for g, a, _ in open_res])
loose = np.mean([f >= 0.5 for _, _, f in open_res])
mc = np.mean(pred[:M] == gold[:M])
print(f"\nsame {M} questions -> multiple choice {mc:.0%} | open-ended, exact match {exact:.0%} | open-ended, token-F1 >= 0.5 {loose:.0%}")

Two things happen at once. First, accuracy collapses: on the same questions where the model picked the right letter fairly often, it
almost never *produces* the right answer by itself. Recognising an answer among four is far easier than recalling it, which is exactly why
HuatuoGPT-o1 removes the options before training with RL (slide 81): the model can no longer guess or eliminate its way to a reward.

Second, grading becomes hard. Exact string match rejects correct answers that are worded differently (*"type 2 pneumocytes"* vs.
*"Type II pneumocytes"*), and token overlap gives partial credit to wrong answers that share a word (*"metabolic acidosis"* vs.
*"metabolic alkalosis"*). Neither is a good **verifier**. HuatuoGPT-o1 therefore uses an LLM verifier that checks whether the final answer
*means* the same as the ground truth, and building reliable rewards for medical RL is a research problem in itself.

## 5 · One more caveat: could the model have seen these questions?
MedQA has been public since 2020, so its questions may be in any recent model's training data. The lecture's fix (slide 43) is to test on
an exam held **after** the model was trained, such as a new licensing exam, or to use live benchmarks that keep adding cases (LiveClin,
slide 140). Treat any score on an old public benchmark as an upper bound.

## Exercises
1. Change the system prompt to *"You are a helpful assistant."* or remove it. How much do accuracy and the predicted-letter distribution change?
2. Replace the rotations with **reversed** option order, or add a fifth option *"E. None of the above"*. Which change hurts the model most?
3. Write a better verifier for section 4: e.g. ask the model itself *"Do these two answers refer to the same thing? yes/no"* and compare
   its verdicts with your own judgment on the 30 pairs. When does an LLM verifier fail?
4. On a T4 GPU, set `MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"` and `N = 300`. Does a 3× larger model become more consistent under rotation,
   or only more accurate?

**Next:** [Notebook 2 — retrieval-augmented answering with citations](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture09/lecture09_2_rag_with_citations.ipynb)